In [27]:
import cortex
import pickle
import numpy as np
import nibabel as nib

In [22]:
atlas_base_path = "/home/zachkaras/fmri/fmri_model/analysis/pipeline/atlases"

# read in 2d mni mask
mask = nib.load(f"{atlas_base_path}/MNI152_T1_2mm_brain_mask.nii.gz")
og_shape = mask.shape
mask = mask.get_fdata().flatten()
brain_idx = np.where(mask>0)[0]

atlas = nib.load(f"{atlas_base_path}/Schaefer2018_400Parcels_7Networks_order_FSLMNI152_2mm.nii.gz")
atlas_vec = atlas.get_fdata().flatten()
atlas_only_brain = atlas_vec[brain_idx]
cortex = np.where(atlas_only_brain != 0)[0]

# Making empty templates to save output
empty_schaefer = np.zeros(atlas_only_brain.shape)
empty_mni = np.zeros(atlas_vec.shape)
    
# def create_histogram(voxcorrs):
#     f = plt.figure(figsize=(8,8))
#     ax = f.add_subplot(1,1,1)
#     ax.hist(voxcorrs, 100) # histogram correlations with 100 bins
#     ax.set_xlabel("Correlation")
#     ax.set_ylabel("Num. voxels");
#     plt.savefig() # TODO

def convert_to_nifti(values):
    # working backwards to save correlation values as voxels in MNI space
    empty_schaefer[cortex] = values
    empty_mni[brain_idx] = empty_schaefer
    result_brain = np.reshape(empty_mni, og_shape)

    # Saving results
    nifti_result = nib.Nifti1Image(result_brain, affine=atlas.affine, header=atlas.header)
    return result_brain, nifti_result
    # nib.save(nifti_result, "test.nii.gz")`

In [23]:
filepath = "/storage1/fmri_model_data/ridge_regression_pca_models/111/codegemma_7b-layer_28-prose-correlations.pkl"
with open(filepath, 'rb') as f:
    data = pickle.load(f)

npy_brain, nifti_brain = convert_to_nifti(data)

In [30]:
# Plot mosaic of correlations
from matplotlib.pyplot import figure, cm
import matplotlib.pyplot as plt
# corrvolume = np.zeros(mask.shape)
# corrvolume[mask>0] = voxcorrs

voxel_vol = cortex.Volume(npy_brain, "test", "fullhead")

# Then we have to get a mapper from voxels to vertices for this transform
mapper = cortex.get_mapper("test", "fullhead", 'line_nearest', recache=True)

# Just pass the voxel data through the mapper to get vertex data
vertex_map = mapper(voxel_vol)

# You can plot both as you would normally plot Volume and Vertex data
cortex.quickshow(voxel_vol)
plt.show()
cortex.quickshow(vertex_map)
plt.show()

# f = figure(figsize=(10,10))
# cortex.mosaic(npy_brain, vmin=0, vmax=0.5, cmap=cm.hot);

# Create a cortex.Volume object from the NumPy array
# You can specify colormap, vmin, vmax, and a description
# volume_data = cortex.Volume(npy_brain, "test", "test",
#                             cmap='viridis', vmin=0, vmax=1,
#                             description='Example NumPy array plot')

# # Display the volume data using quickshow
# cortex.quickshow(volume_data)

# You can also save the visualization as a web page
# cortex.webshow(volume_data, filename='numpy_array_plot.html')

FileNotFoundError: [Errno 2] No such file or directory: '/home/zachkaras/miniconda3/share/pycortex/db/test/transforms/fullhead/matrices.xfm'

In [19]:
# nifti_img = nib.load("/storage1/fmri_model_data/ridge_regression_pca_models/111/codegemma_7b-layer_28-prose-correlations.pkl")
# data = nifti_img.get_fdata()
vol_data = cortex.Volume(nifti)

AttributeError: 'numpy.ndarray' object has no attribute 'Volume'

In [ ]:
import cortex
import nibabel as nib
# Load the NIFTI file: Use nibabel to load your NIFTI image.
# Python

nifti_img = nib.load('your_nifti_file.nii.gz')
data = nifti_img.get_fdata()
# Create a cortex.Volume object: This object encapsulates your volumetric data and its spatial information. You need to provide the data, a subject ID, and a transform name (which defines how the volume aligns with the surface).
# Python

# Assuming 'subject_id' and 'transform_name' are already defined in your pycortex database
# For example, if you have a subject 'S1' and a transform 'func_to_anat'
volume_data = cortex.Volume(data, 'subject_id', 'transform_name')
# Plot the data on the flatmap: Use cortex.quickflat.make_figure to generate a flatmap visualization of your data.
# Python

cortex.quickflat.make_figure(volume_data, with_colorbar=True)